# 05. Clustering

### Objective
Assign each 30-second window to a player archetype using K-Means, then compute soft IDW membership scores and temporal deltas.

### I/O
- **Reads**: `../../data/processed/4_activity_contributions.csv`
- **Writes**: `../../data/processed/5_clustered_telemetry.csv`
- **Writes**: `../../data/processed/cluster_centroids.json`

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
import json, os

INPUT_PATH  = '../../data/processed/4_activity_contributions.csv'
OUTPUT_PATH = '../../data/processed/5_clustered_telemetry.csv'
MODEL_DIR   = '../../data/models'
os.makedirs(MODEL_DIR, exist_ok=True)

print('Libraries imported')

Libraries imported


In [2]:
df = pd.read_csv(INPUT_PATH)
print(f'Loaded {len(df)} rows')

Loaded 3240 rows


Two experiments shaped the clustering decisions made below.

**Experiment A** (`experiments/experiment_A_feature_space_validation/Experiment_A_Complete.ipynb`)
compared clustering on raw normalized features vs. activity percentages (`pct_combat`, `pct_collect`, `pct_explore`)
at K=2,3,4,5 using silhouette, Davies-Bouldin, and Calinski-Harabasz scores.

The actual output showed percentage features produce **higher** silhouette scores at every K value:

| K | pct silhouette | raw silhouette |
|---|---|---|
| 2 | 0.5799 | 0.5058 |
| 3 | **0.4948** | 0.4635 |
| 4 | 0.4856 | 0.3749 |
| 5 | 0.4798 | 0.3117 |

The experiment does confirm the compositional constraint mathematically (pct sums to 1.0),
but empirically the pct space clusters more cleanly because the archetypes ARE defined in
percentage space by design - a Combat player is one with high pct_combat. Raw features add
11-dimensional noise where the archetype signal is diluted. The pct approach is both
practically interpretable (centroids read as "65% combat, 5% collect") and empirically optimal.

**Experiment B** (`experiments/experiment_B_clustering_config/Experiment_B_Complete.ipynb`)
ran a 108-config grid search for raw-feature clustering. Its best config was K=3, 8 features,
minmax_log_sparse, no outlier cap (silhouette=0.3928) - still lower than pct K=3 (0.4948).
The K=3 decision is confirmed: K=2 merges Collect+Explore into one blob, K=4 over-segments.

Because of Experiment A's result, pct features are used below - both interpretable and empirically better separated.

In [3]:
# Archetype pre-filter: keep only windows that unambiguously match one archetype.
# Ambiguous windows (mixed-behaviour) pull the K-Means centroids toward the middle
# of the simplex, degrading separation between Collection and Exploration.
# Dropping them forces the centroids into clean geometric positions before fitting.
#
# Percentile thresholds are computed from the pipeline data so they track
# the actual distribution rather than a hardcoded constant.
#
# NOTE: zero-checks work correctly on normalized columns because MinMaxScaler
# maps raw 0 -> normalized 0. Percentile ordering is preserved by the transform.

p33_dist = df['distanceTraveled'].quantile(0.33)
p66_dist = df['distanceTraveled'].quantile(0.66)

is_collection = (
    (df['itemsCollected'] > 0) &
    (df['distanceTraveled'] < p33_dist) &
    (df['timeInCombat'] == 0)
)
is_combat = (
    (df['enemiesHit'] > 0) &
    (df['kills'] > 0)
)
is_exploration = (
    (df['distanceTraveled'] > p66_dist) &
    (df['itemsCollected'] == 0) &
    (df['enemiesHit'] == 0)
)

is_valid = is_collection | is_combat | is_exploration
n_before = len(df)
df = df[is_valid].copy().reset_index(drop=True)
n_after = len(df)

print(f'Archetype pre-filter: {n_before} → {n_after} rows ({n_before - n_after} ambiguous dropped)')
print(f'  Collection candidates : {is_collection.sum()}')
print(f'  Combat candidates     : {is_combat.sum()}')
print(f'  Exploration candidates: {is_exploration.sum()}')
print(f'  Ambiguous dropped     : {(~is_valid).sum()}')
print(f'  p33_distanceTraveled  = {p33_dist:.4f}')
print(f'  p66_distanceTraveled  = {p66_dist:.4f}')

Archetype pre-filter: 3240 → 1331 rows (1909 ambiguous dropped)
  Collection candidates : 103
  Combat candidates     : 1047
  Exploration candidates: 181
  Ambiguous dropped     : 1909
  p33_distanceTraveled  = 0.3508
  p66_distanceTraveled  = 0.5532


In [4]:
features = ['pct_combat', 'pct_collect', 'pct_explore']
X = df[features].fillna(0)

print(f'Clustering features: {features}')
print(f'Shape: {X.shape}')

Clustering features: ['pct_combat', 'pct_collect', 'pct_explore']
Shape: (1331, 3)


In [5]:
print('Running K-Means (K=3)...')
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)

print('Clustering complete')
print(f'Cluster centers:\n{kmeans.cluster_centers_}')

Running K-Means (K=3)...
Clustering complete
Cluster centers:
[[0.21684694 0.48540925 0.29774381]
 [0.00723067 0.06346497 0.92930436]
 [0.72527203 0.09634007 0.17838791]]


In [6]:
# Map each K-Means cluster to the nearest named archetype using the Hungarian algorithm.
# This gives a stable, deterministic assignment regardless of which cluster index
# K-Means happens to assign on any given run.
ideal_centers = np.array([
    [1.0, 0.0, 0.0],  # ideal Combat
    [0.0, 1.0, 0.0],  # ideal Collection
    [0.0, 0.0, 1.0],  # ideal Exploration
])

distances = cdist(kmeans.cluster_centers_, ideal_centers, metric='euclidean')
row_ind, col_ind = linear_sum_assignment(distances)

archetype_names = ['Combat', 'Collection', 'Exploration']
mapping = {row_ind[i]: archetype_names[col_ind[i]] for i in range(len(row_ind))}

df['archetype'] = df['cluster'].map(mapping)
print(f'Cluster mapping: {mapping}')
print(f'Archetype distribution:\n{df["archetype"].value_counts()}')

Cluster mapping: {np.int64(0): 'Collection', np.int64(1): 'Exploration', np.int64(2): 'Combat'}
Archetype distribution:
archetype
Combat         916
Collection     238
Exploration    177
Name: count, dtype: int64


## Soft Membership Computation

Compute a fractional membership score for each archetype using Inverse Distance Weighting (IDW).
Each window gets a probability-like vector such as `[combat=0.6, collect=0.3, explore=0.1]`
instead of a binary one-hot label. This prevents hard jumps in the difficulty signal when a
player crosses an archetype boundary.

Experiment B Section 2 (`experiments/experiment_B_clustering_config/Experiment_B_Complete.ipynb`)
quantifies the smoothness improvement: soft IDW reduces mean transition magnitude by ~34%
compared to hard one-hot labels.

In [7]:
print('Applying centroids to FULL dataset for MLP training...')

# K-Means was fitted on the filtered 1331 rows to get correct centroid positions.
# IDW soft membership is a distance formula — it is valid for any window, including
# mixed-behaviour sessions. Reloading the original 3240 rows here so the MLP trains
# on full input-space coverage, not just archetype-dominant examples.
df_full = pd.read_csv(INPUT_PATH)
features = ['pct_combat', 'pct_collect', 'pct_explore']
X_full = df_full[features].fillna(0)

# Distance from every window to each centroid
distances_full = kmeans.transform(X_full)

# Invert distances — closer centroid = higher membership
inv_distances_full = 1 / (distances_full + 1e-10)

# Normalize so all three scores sum to 1.0 per window
soft_membership_full = inv_distances_full / inv_distances_full.sum(axis=1, keepdims=True)

combat_idx  = [k for k, v in mapping.items() if v == 'Combat'][0]
collect_idx = [k for k, v in mapping.items() if v == 'Collection'][0]
explore_idx = [k for k, v in mapping.items() if v == 'Exploration'][0]

df_full['soft_combat']  = soft_membership_full[:, combat_idx]
df_full['soft_collect'] = soft_membership_full[:, collect_idx]
df_full['soft_explore'] = soft_membership_full[:, explore_idx]

df_full['cluster']   = kmeans.predict(X_full)
df_full['archetype'] = df_full['cluster'].map(mapping)

print(f'Soft membership applied to {len(df_full)} rows (full dataset)')
print(f'Mean soft_combat:  {df_full["soft_combat"].mean():.4f}')
print(f'Mean soft_collect: {df_full["soft_collect"].mean():.4f}')
print(f'Mean soft_explore: {df_full["soft_explore"].mean():.4f}')

Applying centroids to FULL dataset for MLP training...
Soft membership applied to 3240 rows (full dataset)
Mean soft_combat:  0.3815
Mean soft_collect: 0.3301
Mean soft_explore: 0.2884


## Temporal Delta Computation

Calculate window-to-window change in soft membership scores per player.
A positive delta_combat means the player is becoming more combat-focused this window.
A negative delta means they are shifting away. These deltas are the primary variance
driver in the ANFIS target variable - see notebook 06 and Experiment C for why.

In [8]:
print('Computing temporal deltas...')

# Sort within each player's timeline before differencing.
# groupby('userId') ensures we never subtract User B's first window from User A's last.
df_sorted = df_full.sort_values(['userId', 'timestamp'])

df_sorted['delta_combat']  = df_sorted.groupby('userId')['soft_combat'].diff().fillna(0)
df_sorted['delta_collect'] = df_sorted.groupby('userId')['soft_collect'].diff().fillna(0)
df_sorted['delta_explore'] = df_sorted.groupby('userId')['soft_explore'].diff().fillna(0)

df_full = df_sorted.copy()

print('Deltas computed (first window per player = 0)')
print(f'delta_combat range:  [{df_full["delta_combat"].min():.3f}, {df_full["delta_combat"].max():.3f}]')
print(f'delta_collect range: [{df_full["delta_collect"].min():.3f}, {df_full["delta_collect"].max():.3f}]')
print(f'delta_explore range: [{df_full["delta_explore"].min():.3f}, {df_full["delta_explore"].max():.3f}]')

Computing temporal deltas...
Deltas computed (first window per player = 0)
delta_combat range:  [-0.924, 0.906]
delta_collect range: [-0.792, 0.785]
delta_explore range: [-0.893, 0.902]


In [9]:
df_full.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to {OUTPUT_PATH}')
print(f'Total rows: {len(df_full)} (K-Means centroids from 1331 filtered rows, soft membership applied to all)')
print('Clustering + soft membership + deltas complete!')

Saved to ../../data/processed/5_clustered_telemetry.csv
Total rows: 3240 (K-Means centroids from 1331 filtered rows, soft membership applied to all)
Clustering + soft membership + deltas complete!


In [10]:
# Export cluster centroids for the demo UI.
# The TypeScript engine uses these to recompute IDW membership for live player input.
CENTROIDS_OUTPUT = os.path.join(MODEL_DIR, 'cluster_centroids.json')
CENTROIDS_DEPLOY = '../../../anfis-demo-ui/models/cluster_centroids.json'

centroids_export = {}
for cluster_id, archetype_name in mapping.items():
    center = kmeans.cluster_centers_[cluster_id]
    centroids_export[str(cluster_id)] = {
        'archetype': archetype_name,
        'centroid': {
            'pct_combat':  float(center[0]),
            'pct_collect': float(center[1]),
            'pct_explore': float(center[2])
        }
    }

with open(CENTROIDS_OUTPUT, 'w') as f:
    json.dump(centroids_export, f, indent=2)

try:
    os.makedirs(os.path.dirname(CENTROIDS_DEPLOY), exist_ok=True)
    with open(CENTROIDS_DEPLOY, 'w') as f:
        json.dump(centroids_export, f, indent=2)
    print(f'Deployed to: {CENTROIDS_DEPLOY}')
except Exception:
    print('Deploy path not found - skipping deploy copy')

print(f'Saved to: {CENTROIDS_OUTPUT}')
print(json.dumps(centroids_export, indent=2))

Deployed to: ../../../anfis-demo-ui/models/cluster_centroids.json
Saved to: ../../data/models\cluster_centroids.json
{
  "0": {
    "archetype": "Collection",
    "centroid": {
      "pct_combat": 0.2168469378621355,
      "pct_collect": 0.4854092502249181,
      "pct_explore": 0.29774381191294647
    }
  },
  "1": {
    "archetype": "Exploration",
    "centroid": {
      "pct_combat": 0.007230672593310339,
      "pct_collect": 0.06346497116196423,
      "pct_explore": 0.9293043562447255
    }
  },
  "2": {
    "archetype": "Combat",
    "centroid": {
      "pct_combat": 0.7252720260429242,
      "pct_collect": 0.09634006886112138,
      "pct_explore": 0.17838790509595437
    }
  }
}
